# Calibration inspection notebook

Ad-hoc verification of the nightly channel-snapped emission calibration
(job: `emissions_calibration_job`).

The inverted Q field is consumed by `gaussian_forward_forecast_detailed`,
which runs a **24-hour** forecast at **15-minute cadence** using
`modeldata_forecast_15min.parquet` (`FORECAST_DATA_15MIN_PATH`) as its
meteorological driver.

Reads straight from S3:

- `tijuana/dispersion/calibration/Q_field_latest.parquet` — per-segment emission rates
- `tijuana/dispersion/calibration/inversion_diagnostics_latest.json` — CV + budget gates
- `latest/tijuana/dispersion/forward_forecast_latest.json` *(optional)* — most recent 24h × 15-min forecast

Produces the same verification plots as the `calibration_viz` Dagster
asset, but interactive and throwaway. Use this for one-off debugging;
the PNGs on S3 (`.../calibration/viz/*_latest.png`) are the canonical
nightly artifacts.

## Prerequisites

```bash
# From repo root — reuse the h2s project venv so pandas/matplotlib/boto3 are all there
cd projects/h2s
uv sync
uv run python -m ipykernel install --user --name h2s --display-name "h2s (uv)"
# Then: jupyter lab  (select the 'h2s (uv)' kernel)
```

A `.env` file at repo root with `S3_*` credentials is loaded automatically.

In [ ]:
import io
import json
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from botocore.client import Config

# Load .env from repo root (optional; harmless if python-dotenv missing)
try:
    from dotenv import load_dotenv
    load_dotenv(Path(os.getcwd()).resolve().parent / "projects" / "h2s" / ".env")
    load_dotenv(Path(os.getcwd()).resolve().parent / ".env")
except ImportError:
    pass

S3_BUCKET     = os.environ["S3_BUCKET"]
S3_ADDRESS    = os.environ["S3_ADDRESS"]
S3_PORT       = os.environ.get("S3_PORT", "443")
S3_USE_SSL    = os.environ.get("S3_USE_SSL", "true").lower() == "true"
S3_ACCESS_KEY = os.environ["S3_ACCESS_KEY"]
S3_SECRET_KEY = os.environ["S3_SECRET_KEY"]

scheme = "https" if S3_USE_SSL else "http"
endpoint = f"{scheme}://{S3_ADDRESS}:{S3_PORT}"

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    config=Config(signature_version="s3v4"),
)

print(f"S3 endpoint: {endpoint}  bucket: {S3_BUCKET}")

In [ ]:
CALIBRATION_BASE = "tijuana/dispersion/calibration"
Q_FIELD_LATEST = f"{CALIBRATION_BASE}/Q_field_latest.parquet"
DIAG_LATEST    = f"{CALIBRATION_BASE}/inversion_diagnostics_latest.json"

def s3_get_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=S3_BUCKET, Key=key)
    return obj["Body"].read()

q_df = pd.read_parquet(io.BytesIO(s3_get_bytes(Q_FIELD_LATEST)))
diagnostics = json.loads(s3_get_bytes(DIAG_LATEST))

print(f"Q_field rows: {len(q_df)}  cols: {list(q_df.columns)}")
print(f"Diagnostics run timestamp: {diagnostics.get('timestamp')}")
q_df.head()

## Summary stats

In [ ]:
active = q_df[q_df["Q_g_s"] > 0]
q_total = float(q_df["Q_g_s"].sum())
gates = diagnostics.get("gates", {})

print(f"Total segments:        {len(q_df)}")
print(f"Active (Q > 0):        {len(active)}")
print(f"\u03a3 Q (g/s):           {q_total:.2f}")
print(f"Max segment Q:         {q_df['Q_g_s'].max():.2f} g/s")
print(f"Median active Q:       {active['Q_g_s'].median():.3f} g/s"
      if not active.empty else "Median active Q:       n/a")
print()
print("Gates:")
for k, v in gates.items():
    print(f"  {k}: {v}")

# Top-5 emitting segments
print("\nTop 10 active segments:")
active.nlargest(10, "Q_g_s")[["segment_idx", "lat", "lon", "Q_g_s"]]

## Channel map — segments colored by Q

In [ ]:
SENSORS = {
    "NESTOR - BES": {"lat": 32.545, "lon": -117.075},
    "IB CIVIC CTR": {"lat": 32.5586, "lon": -117.0914},
    "SAN YSIDRO":   {"lat": 32.5552, "lon": -117.0386},
}

fig, ax = plt.subplots(figsize=(10, 8))
inactive = q_df[q_df["Q_g_s"] == 0]
ax.scatter(inactive["lon"], inactive["lat"], c="lightgray", s=18, alpha=0.6)

if not active.empty:
    sc = ax.scatter(
        active["lon"], active["lat"],
        c=active["Q_g_s"], s=60 + 8 * active["Q_g_s"],
        cmap="YlOrRd", edgecolor="black", linewidth=0.3, alpha=0.9,
    )
    plt.colorbar(sc, ax=ax, label="Q (g/s)")

for sname, sloc in SENSORS.items():
    ax.plot(sloc["lon"], sloc["lat"], marker="^", markersize=16,
            color="blue", markeredgecolor="white")
    ax.annotate(sname, (sloc["lon"], sloc["lat"]),
                xytext=(6, 6), textcoords="offset points",
                fontsize=9, color="blue")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"Channel Q field \u2014 {len(active)}/{len(q_df)} active, "
    f"\u03a3 Q = {q_total:.1f} g/s"
)
ax.set_aspect("equal", adjustable="datalim")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Leave-one-sensor-out CV scatter

In [ ]:
loo = diagnostics.get("leave_one_sensor_out", {}) or {}

fig, ax = plt.subplots(figsize=(7, 7))
colors = {"NESTOR - BES": "tab:blue", "IB CIVIC CTR": "tab:orange", "SAN YSIDRO": "tab:green"}
max_val = 1.0
for sname, data in loo.items():
    obs = data.get("c_obs_ppb") or []
    pred = data.get("c_pred_ppb") or []
    if not obs or not pred:
        continue
    rmse = data.get("rmse_ppb")
    bias = data.get("bias_ppb")
    ax.scatter(obs, pred, s=42, alpha=0.75,
               c=colors.get(sname, "tab:gray"),
               edgecolor="black", linewidth=0.3,
               label=f"{sname} (RMSE={rmse}, bias={bias})")
    max_val = max(max_val, max(obs + pred))
lim = max_val * 1.1
ax.plot([0, lim], [0, lim], color="black", linestyle="--", linewidth=1, label="1:1")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Observed C (ppb, held-out sensor)")
ax.set_ylabel("Predicted C (ppb, from other sensors)")
ax.set_title(
    f"Leave-one-sensor-out CV \u2014 gate pass: "
    f"{gates.get('leave_one_sensor_out_pass', False)}"
)
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## Budget bar — Σ Q vs allowed range

In [ ]:
budget = diagnostics.get("budget_sanity", {}) or {}
low  = float(budget.get("allowed_low", 30.0))
high = float(budget.get("allowed_high", 500.0))
anchor = float(budget.get("anchor_g_s", 167.0))
gate_pass = gates.get("budget_sanity_pass", False)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.axvspan(low, high, color="lightgreen", alpha=0.35, label="allowed")
ax.axvline(anchor, color="gray", linestyle="--", linewidth=1.5,
           label=f"anchor ({anchor:.0f} g/s)")
ax.axvline(q_total, color="green" if gate_pass else "red",
           linewidth=3.5, label=f"\u03a3 Q = {q_total:.1f} g/s")
ax.set_xlim(0, max(high * 1.1, q_total * 1.1))
ax.set_xlabel("\u03a3 Q (g/s)")
ax.set_yticks([])
ax.set_title(f"Budget sanity \u2014 gate pass: {gate_pass}")
ax.grid(True, axis="x", alpha=0.3)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## Q distribution — histogram (active segments only)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
if not active.empty:
    ax.hist(active["Q_g_s"], bins=30, color="#d95f02", edgecolor="black", alpha=0.8)
ax.set_xlabel("Q (g/s) per active segment")
ax.set_ylabel("Count")
ax.set_title(f"Active segment Q distribution (n={len(active)})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Raw diagnostics JSON (expandable)

In [ ]:
print(json.dumps(
    {k: v for k, v in diagnostics.items() if k != "leave_one_sensor_out"},
    indent=2, default=str,
))
print("\n# leave_one_sensor_out (c_obs/c_pred arrays omitted for brevity)")
print(json.dumps(
    {k: {kk: vv for kk, vv in v.items() if kk not in ("c_obs_ppb", "c_pred_ppb")}
     for k, v in diagnostics.get("leave_one_sensor_out", {}).items()},
    indent=2, default=str,
))

## Optional: latest 24h × 15-min forward forecast driven by this Q field

Pulls the most recent `forward_forecast_latest.json` and plots per-sensor
timeseries. Confirms that the Q field you just inspected actually drives
downstream predictions.

In [ ]:
FORECAST_LATEST = "latest/tijuana/dispersion/forward_forecast_latest.json"
try:
    fc = json.loads(s3_get_bytes(FORECAST_LATEST))
except Exception as exc:
    print(f"No forecast at {FORECAST_LATEST}: {exc}")
    fc = None

if fc:
    times = pd.to_datetime(fc["times"])
    meta = fc.get("metadata", {})
    print(f"Forecast metadata: hours={meta.get('hours')} cadence_minutes={meta.get('cadence_minutes')}")
    print(f"Source mode: {meta.get('model', meta.get('source_mode', 'n/a'))}")
    print(f"Frames: {len(times)}  (expected: {meta.get('hours', 24)*60 // meta.get('cadence_minutes', 15)})")

    fig, ax = plt.subplots(figsize=(12, 4))
    for sname, series in fc["concentrations"].items():
        ax.plot(times, series, label=sname, linewidth=1.4, marker=".", markersize=3)
    ax.axhline(30, color="orange", linestyle="--", linewidth=1, alpha=0.7, label="30 ppb WATCH")
    ax.axhline(100, color="red", linestyle="--", linewidth=1, alpha=0.7, label="100 ppb CRITICAL")
    ax.set_xlabel("Time")
    ax.set_ylabel("H2S (ppb)")
    ax.set_title("Latest 24h x 15-min forward forecast per sensor")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()